In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("\n" + "#"*20)
print("SCHEMA B GLOVE RF ERROR ANALYSIS")
print("#"*20)

labels = encoder.classes_

cm = confusion_matrix(y_test_b, y_pred_b, labels=np.arange(len(labels)))
print("\nClassification report for Schema B GloVe RF:")
print(classification_report(y_test_b, y_pred_b, target_names=labels))

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Schema B GloVe RF Confusion Matrix")
plt.tight_layout()
plt.show()

error_df = pd.DataFrame({
    "review_text": df_truth_aligned.iloc[idx_test_b]["review_text"].values,
    "true_label": encoder.inverse_transform(y_test_b),
    "predicted_label": encoder.inverse_transform(y_pred_b)
})

error_df["correct"] = error_df["true_label"] == error_df["predicted_label"]
error_df["review_length"] = error_df["review_text"].astype(str).str.len()
error_df["misclass_pair"] = error_df["true_label"] + " -> " + error_df["predicted_label"]

misclassified = error_df[error_df["correct"] == False].copy()

print(f"Total test samples: {len(error_df)}")
print(f"Misclassified samples: {len(misclassified)}")
print(f"Misclassification rate: {len(misclassified) / len(error_df):.2%}")

pair_counts = misclassified["misclass_pair"].value_counts()
print("\nTop misclassification pairs:")
print(pair_counts.head(10).to_string())

print("\nReview-length statistics for misclassified samples:")
print(misclassified["review_length"].describe())

save_path = "schema_b_glove_rf_misclassified.csv"
misclassified.to_csv(save_path, index=False)
print(f"\nSaved all misclassified reviews to: {save_path}")

print("\nMisclassified label distribution:")
print(misclassified["true_label"].value_counts().to_string())
print("\nPredicted label distribution for misclassified samples:")
print(misclassified["predicted_label"].value_counts().to_string())

pair_pct = (misclassified["misclass_pair"].value_counts(normalize=True) * 100).round(2)
print("\nMisclassification pair percentages:")
print(pair_pct.head(10).to_string())

# Simple term frequency analysis on misclassified reviews
from collections import Counter
import re

all_text = " ".join(misclassified["review_text"].astype(str).tolist()).lower()
words = re.findall(r"\b[a-z]{3,}\b", all_text)
stopwords = set(["the","and","for","with","this","that","from","have","has","not","but","you","your","game","games"])
word_counts = Counter([w for w in words if w not in stopwords])
print("\nTop words in misclassified reviews:")
for word, count in word_counts.most_common(20):
    print(f"{word}: {count}")

print("\nSample misclassified reviews:")
print(misclassified[["review_text", "true_label", "predicted_label", "review_length"]].head(15).to_string(index=False))

In [ ]:
print(misclassified)

In [ ]:
import pandas as pd

def categorize_errors(row):
    text = row['review_text'].lower()
    
    # Pattern 1: Sarcasm/Irony
    if "gotta love" in text or "i love... sadly" in text:
        return "Sarcasm/Irony"
    
    # Pattern 2: Contrastive Adjectives (The "But" Pivot)
    if "but" in text or "tho" in text or "although" in text:
        return "Contrastive Logic"
    
    # Pattern 3: Gaming Slang/Memes
    memes = ["try fingers", "super earth", "monster my hunter"]
    if any(m in text for m in memes):
        return "Domain Slang/Memes"
    
    # Pattern 4: Length/Noise
    if row['review_length'] > 500:
        return "Long-form Noise"
    
    return "Uncategorized"

# Assuming 'errors_df' is your table above
misclassified['failure_reason'] = misclassified.apply(categorize_errors, axis=1)

# Summary for your report
summary = misclassified['failure_reason'].value_counts()
print(summary)

In [ ]:
# visualization using plotly
import matplotlib.pyplot as plt 
plt.figure(figsize=(8,5))
summary.plot(kind='bar', color=['orange', 'blue', 'green', 'red'])  
plt.title('Error Categorization of Misclassified Reviews')
plt.xlabel('Failure Reason')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

def categorize_errors(row):
    text = row['review_text'].lower()
    
    # Pattern 1: Sarcasm/Irony
    if "gotta love" in text or "i love... sadly" in text:
        return "Sarcasm/Irony"
    
    # Pattern 2: Contrastive Adjectives (The "But" Pivot)
    if "but" in text or "tho" in text or "although" in text:
        return "Contrastive Logic"
    
    # Pattern 3: Gaming Slang/Memes
    memes = ["try fingers", "super earth", "monster my hunter"]
    if any(m in text for m in memes):
        return "Domain Slang/Memes"
    
    # Pattern 4: Length/Noise
    if row['review_length'] > 500:
        return "Long-form Noise"
    
    return "Uncategorized"

# Assuming 'errors_df' is your table above
misclassified['failure_reason'] = misclassified.apply(categorize_errors, axis=1)

# Summary for your report
summary = misclassified['failure_reason'].value_counts()
print(summary)

---

In [ ]:
print("\n" + "#" * 20)
print("SCHEMA B GLOVE RF ERROR ANALYSIS (OPTIMIZED)")
print("#" * 20)

labels = encoder.classes_

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(labels)))
print("\nClassification report for Schema B GloVe RF (Optimized):")
print(classification_report(y_test, y_pred, target_names=labels))

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Schema B GloVe RF Confusion Matrix (Optimized)")
plt.tight_layout()
plt.show()

# ─── BUILD ERROR DATAFRAME ─────────────────────────────────────────────────────
# Retrieve the test-set rows from the aligned dataframe using the same split
_, idx_test = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

error_df = pd.DataFrame({
    "review_text": df_truth_aligned.iloc[idx_test]["review_text"].values,
    "true_label": encoder.inverse_transform(y_test),
    "predicted_label": encoder.inverse_transform(y_pred)
})

error_df["correct"] = error_df["true_label"] == error_df["predicted_label"]
error_df["review_length"] = error_df["review_text"].astype(str).str.len()
error_df["misclass_pair"] = error_df["true_label"] + " -> " + error_df["predicted_label"]

misclassified = error_df[error_df["correct"] == False].copy()

print(f"\nTotal test samples: {len(error_df)}")
print(f"Misclassified samples: {len(misclassified)}")
print(f"Misclassification rate: {len(misclassified) / len(error_df):.2%}")

pair_counts = misclassified["misclass_pair"].value_counts()
print("\nTop misclassification pairs:")
print(pair_counts.head(10).to_string())

print("\nReview-length statistics for misclassified samples:")
print(misclassified["review_length"].describe())

save_path = "schema_b_glove_rf_optimized_misclassified.csv"
misclassified.to_csv(save_path, index=False)
print(f"\nSaved all misclassified reviews to: {save_path}")

print("\nMisclassified label distribution:")
print(misclassified["true_label"].value_counts().to_string())
print("\nPredicted label distribution for misclassified samples:")
print(misclassified["predicted_label"].value_counts().to_string())

pair_pct = (misclassified["misclass_pair"].value_counts(normalize=True) * 100).round(2)
print("\nMisclassification pair percentages:")
print(pair_pct.head(10).to_string())

# ─── TERM FREQUENCY ANALYSIS ───────────────────────────────────────────────────
all_text = " ".join(misclassified["review_text"].astype(str).tolist()).lower()
words = re.findall(r"\b[a-z]{3,}\b", all_text)
stopwords = set(["the", "and", "for", "with", "this", "that", "from", "have", "has", "not", "but", "you", "your", "game", "games"])
word_counts = Counter([w for w in words if w not in stopwords])
print("\nTop words in misclassified reviews:")
for word, count in word_counts.most_common(20):
    print(f"{word}: {count}")

print("\nSample misclassified reviews:")
print(misclassified[["review_text", "true_label", "predicted_label", "review_length"]].head(15).to_string(index=False))

In [ ]:
print(misclassified)

In [ ]:
# ─── FAILURE REASON CATEGORIZATION ────────────────────────────────────────────
import pandas as pd

def categorize_errors(row):
    text = str(row['review_text']).lower()
    true  = str(row['true_label']).lower()
    pred  = str(row['predicted_label']).lower()

    # Pattern 1: Sarcasm / Irony
    sarcasm_markers = [
        "gotta love", "i love... sadly", "what a game", "wow amazing",
        "so fun", "totally worth", "best game ever", "couldn't be better"
    ]
    if any(m in text for m in sarcasm_markers):
        return "Sarcasm/Irony"

    # Pattern 2: Contrastive Logic ("But" pivot)
    contrast_markers = ["but", "tho", "although", "however", "yet", "despite", "even though"]
    if any(m in text for m in contrast_markers):
        return "Contrastive Logic"

    # Pattern 3: Domain Slang / Memes
    memes = [
        "try fingers", "super earth", "monster my hunter",
        "git gud", "skill issue", "cope", "seethe", "gg ez",
        "no cap", "based", "mid game", "goated", "slaps"
    ]
    if any(m in text for m in memes):
        return "Domain Slang/Memes"

    # Pattern 4: Long-form Noise
    if row['review_length'] > 500:
        return "Long-form Noise"

    # Pattern 5: Very Short / Low-signal reviews
    if row['review_length'] < 30:
        return "Too Short / Low Signal"

    # Pattern 6: Numeric / Symbol Heavy (e.g. "10/10", "5 stars ★★★")
    import re
    if re.search(r'\d+/\d+|★|☆|[0-9]+\s*(stars?|out of)', text):
        return "Numeric Rating Language"

    # Pattern 7: Boundary confusion (adjacent classes mixed up)
    boundary_pairs = [
        ("positive", "very positive"),
        ("negative", "very negative"),
        ("very positive", "positive"),
        ("very negative", "negative"),
    ]
    if (true, pred) in boundary_pairs:
        return "Boundary Class Confusion"

    return "Uncategorized"


# ─── APPLY & REPORT ───────────────────────────────────────────────────────────
misclassified['failure_reason'] = misclassified.apply(categorize_errors, axis=1)

summary = misclassified['failure_reason'].value_counts()
summary_pct = (misclassified['failure_reason'].value_counts(normalize=True) * 100).round(2)

print("\n" + "#" * 20)
print("FAILURE REASON ANALYSIS")
print("#" * 20)
print(f"\n{'Failure Reason':<30} {'Count':>6}  {'%':>6}")
print("-" * 45)
for reason in summary.index:
    print(f"{reason:<30} {summary[reason]:>6}  {summary_pct[reason]:>5.1f}%")

# Per-reason breakdown: which true labels are most affected
print("\nFailure reason × true label breakdown:")
breakdown = (
    misclassified.groupby(['failure_reason', 'true_label'])
    .size()
    .reset_index(name='count')
    .sort_values(['failure_reason', 'count'], ascending=[True, False])
)
print(breakdown.to_string(index=False))

# Save enriched misclassified file
save_path = "schema_b_glove_rf_optimized_misclassified_categorized.csv"
misclassified.to_csv(save_path, index=False)
print(f"\nSaved categorized misclassifications to: {save_path}")

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Failure reasons",
        "Top misclass pairs",
        "True label distribution",
        "Review length by outcome",
    ]
)

# Failure reasons
fr = misclassified["failure_reason"].value_counts().reset_index()
fig.add_trace(go.Bar(x=fr["count"], y=fr["failure_reason"], orientation="h", showlegend=False), row=1, col=1)

# Misclass pairs
pc = misclassified["misclass_pair"].value_counts().head(8).reset_index()
fig.add_trace(go.Bar(x=pc["count"], y=pc["misclass_pair"], orientation="h", showlegend=False), row=1, col=2)

# True label dist
tl = misclassified["true_label"].value_counts().reset_index()
fig.add_trace(go.Bar(x=tl["true_label"], y=tl["count"], showlegend=False), row=2, col=1)

# Length box
for name, grp in error_df.groupby("correct"):
    fig.add_trace(go.Box(y=grp["review_length"], name="Correct" if name else "Wrong", showlegend=True), row=2, col=2)

fig.update_layout(height=700, title_text="GloVe RF — Error Analysis", template="plotly_white")
fig.show()